In [1]:
import cv2
import numpy as np
import os

In [ ]:
# 1. White Balance
def white_balance(img):
    img = img.astype(np.float32)
    avg_b = np.mean(img[:,:,0])
    avg_g = np.mean(img[:,:,1])
    avg_r = np.mean(img[:,:,2])

    avg_gray = (avg_b + avg_g + avg_r) / 3

    img[:,:,0] *= (avg_gray / (avg_b + 1e-6))
    img[:,:,1] *= (avg_gray / (avg_g + 1e-6))
    img[:,:,2] *= (avg_gray / (avg_r + 1e-6))

    return np.clip(img, 0, 255)

# 2. Estimate Backscatter
def estimate_backscatter(img, kernel_size=31):
    B = cv2.blur(img, (kernel_size, kernel_size))
    return B


# 3. Estimate Transmission
# Red Channel Prior 
def estimate_transmission(img, B, omega=0.95):
    img = img / 255.0
    B = B / 255.0
    B = np.maximum(B, 0.05)
    t = 1 - omega * (img[:,:,2] / B[:,:,2])

    return np.clip(t, 0.05, 1.0)


# 4. Refine Transmission
def refine_transmission(t, img):
    gray = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    t_refined = cv2.bilateralFilter(t.astype(np.float32), 9, 75, 75)
    return np.clip(t_refined, 0.05, 1.0)


# 5. Recover Scene Radiance
def recover_image(img, B, t, t_min=0.1):
    img = img.astype(np.float32)
    B = B.astype(np.float32)
    t = np.maximum(t, t_min)
    t = np.repeat(t[:, :, np.newaxis], 3, axis=2)
    J = (img - B) / t + B
    return np.clip(J, 0, 255)


# 6. Wavelength Compensation
def compensate_wavelength(J):
    J = J.astype(np.float32)
    mean_r = np.mean(J[:,:,2])
    mean_g = np.mean(J[:,:,1])
    mean_b = np.mean(J[:,:,0])
    if mean_r < mean_g:
        J[:,:,2] *= 1.3
    if mean_r < mean_b:
        J[:,:,2] *= 1.3

    if mean_g < mean_b:
        J[:,:,1] *= 1.1

    return np.clip(J, 0, 255)


# 7. CLAHE
def apply_clahe(img):
    img = img.astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])

    result = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return result


def enhance_underwater_image(input_path, output_path):
    img = cv2.imread(input_path).astype(np.float32)

    wb = white_balance(img)
    B = estimate_backscatter(wb)
    t = estimate_transmission(wb, B)
    t_refined = refine_transmission(t, wb)
    recovered = recover_image(wb, B, t_refined)
    compensated = compensate_wavelength(recovered)
    final = apply_clahe(compensated)
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, final)

    return final

if __name__ == "__main__":
    input_img = "input.jpg"
    output_img = "output/enhanced.jpg"

    enhance_underwater_image(input_img, output_img)